## google/paligemma2-3b-mix-224  
    Final Overall Accuracy: 67.59%  
    Per-task Accuracy:  
    Count: 60.66% (478/788)  
    Relation: 76.00% (494/650)


In [1]:
from src.models import DinoVLM
import torch
from icecream import ic

from transformers import pipeline
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from src.dataset_utils import *
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import os

Failed to load /home/system/miniconda3/envs/gemma/lib/python3.10/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/system/miniconda3/envs/gemma/lib/python3.10/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /home/system/miniconda3/envs/gemma/lib/python3.10/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/system/miniconda3/envs/gemma/lib/python3.10/site-packages/torchao/_C_cutlass_90a.abi3.so


In [2]:
model = DinoVLM()
processor = model.processor
model.freeze_lm()
model.freeze_vision()

total_trainable_params = sum([p.numel() for p in model.parameters() if p.requires_grad])
print(f'[INFO] Total trainable parameters: {total_trainable_params}')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using cache found in /home/system/.cache/torch/hub/ywyue_FiT3D_main
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[INFO] Freezed LM
[INFO] Freezed ViT
[INFO] Total trainable parameters: 887040


In [3]:
import json

def collate_fn(examples):
    texts  = []
    images = []

    for example in examples:
        messages = example["messages"]
        
        system_text = messages[0]["content"][0]["text"]
        image       = messages[1]["content"][0]["image"]   # PIL image
        user_text   = messages[1]["content"][1]["text"]
        answer      = messages[2]["content"][0]["text"]    # dict or string
        
        # answer might be a dict, convert to json string
        if isinstance(answer, dict):
            answer = json.dumps(answer)
        
        # flatten: system + user prompt + answer
        full_text = f"{system_text}\n{user_text}\n{answer}"
        
        texts.append(full_text)
        images.append(image)

    batch_input = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    labels = batch_input["input_ids"].clone()
    image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    batch_input["labels"] = labels

    return batch_input

In [4]:
dataset = load_dataset('mrdbourke/FoodExtract-1k-Vision')
# dataset['train'][0]
dataset_split = dataset['train'].train_test_split(test_size = 0.1)
train_dataset = [format_data(sample) for sample in dataset_split["train"]]
test_dataset = [format_data(sample) for sample in dataset_split["test"]]

train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle = True, collate_fn=collate_fn)
test_dataset = DataLoader(test_dataset, batch_size=1, shuffle = True, collate_fn=collate_fn)


In [5]:
# collate_fn([train_dataset[0]])
train_dataset[0]

{'messages': [{'role': 'system',
   'content': [{'type': 'text',
     'text': 'You are an expert food and drink image extractor.\nYou provide structured data to visual inputs classifying them as edible food/drink or not.\nAs well as titling the image with a simple food/drink related caption.\nFinally you extract any and all visible food/drink items to lists.\n'}]},
  {'role': 'user',
   'content': [{'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=512x306>},
    {'type': 'text',
     'text': "Classify the given input image into food or not and if edible food or drink items are present, extract those to a list. If no food/drink items are visible, return empty lists.\n\nOnly return valid JSON in the following form:\n\n```json\n{\n  'is_food': 0, # int - 0 or 1 based on whether food/drinks are present (0 = no foods visible, 1 = foods visible)\n  'image_title': '', # str - short food-related title for what foods/drinks are visible in the image, leave blank if no foods pr

In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=r"language_model\.layers\.\d+\.self_attn\.(q|k|v)_proj",
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,579,328 || all params: 2,644,666,112 || trainable%: 0.1732


In [7]:
NUM_EPOCHS      = 4
GRAD_ACCUM_STEPS = 4
LR              = 2e-4
MAX_GRAD_NORM   = 1.0
SAVE_DIR        = "checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)
scheduler = CosineAnnealingLR(optimizer, T_max=len(train_dataloader) * NUM_EPOCHS)
scaler    = torch.cuda.amp.GradScaler()

best_eval_loss = float("inf")

/tmp/ipykernel_426143/3759980505.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


In [8]:
device = next(model.parameters()).device
print(f"Model is on device: {device}")

Model is on device: cuda:0


In [18]:

for b in train_dataloader:
    for key, data in b.items():
        print(key, data.shape)
    # print(b['attention_mask'])
    # print(b['input_ids'])
    break

You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


input_ids torch.Size([1, 520])
attention_mask torch.Size([1, 520])
pixel_values torch.Size([1, 3, 224, 224])
labels torch.Size([1, 520])


In [ ]:

device = 'cuda'
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss   = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss    = outputs.loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        train_loss += outputs.loss.item()  # log unscaled loss

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if step % 10 == 0:
            avg = train_loss / (step + 1)
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Step {step}/{len(train_dataloader)} | Loss {avg:.4f}")

    # handle leftover steps if dataset isn't divisible by GRAD_ACCUM_STEPS
    if (step + 1) % GRAD_ACCUM_STEPS != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    avg_train_loss = train_loss / len(train_dataloader)
    print(f"\nEpoch {epoch+1} complete | Avg train loss: {avg_train_loss:.4f}")

    model.eval()
    eval_loss = 0.0

    with torch.no_grad():
        for batch in test_dataset:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**batch)
            eval_loss += outputs.loss.item()

    avg_eval_loss = eval_loss / len(test_dataset)
    print(f"Epoch {epoch+1} | Eval loss: {avg_eval_loss:.4f}")

    # ── Save best ─────────────────────────────────────────────────────────────
    if avg_eval_loss < best_eval_loss:
        best_eval_loss = avg_eval_loss
        # saves only LoRA + connector weights, not the frozen LM/ViT
        model.save_pretrained(os.path.join(SAVE_DIR, "best"))
        print(f"  ↳ New best saved (eval_loss={avg_eval_loss:.4f})")

    # save every epoch as a checkpoint too
    model.save_pretrained(os.path.join(SAVE_DIR, f"epoch_{epoch+1}"))

print(f"\nTraining done. Best eval loss: {best_eval_loss:.4f}")

You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


Epoch 1/4 | Step 0/1359 | Loss 9.7280


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
Y

Epoch 1/4 | Step 10/1359 | Loss 9.0559


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
Y

...

Epoch 1/4 | Step 60/1359 | Loss 5.8285


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
Y

Epoch 1/4 | Step 70/1359 | Loss 5.4082


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
Y

KeyboardInterrupt: 